In [1]:
import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder

#Load dữ liệu
df = pd.read_csv(r"D:\data_mining\tuan4\data.csv", header=None)
print(df)

       0      1      2       3     4      5
0   Wine  Chips  Bread  Butter  Milk  Apple
1   Wine    NaN  Bread  Butter  Milk    NaN
2    NaN    NaN  Bread  Butter  Milk    NaN
3    NaN  Chips    NaN     NaN   NaN  Apple
4   Wine  Chips  Bread  Butter  Milk  Apple
5   Wine  Chips    NaN     NaN  Milk    NaN
6   Wine  Chips  Bread  Butter   NaN  Apple
7   Wine  Chips    NaN     NaN  Milk    NaN
8   Wine    NaN  Bread     NaN   NaN  Apple
9   Wine    NaN  Bread  Butter  Milk    NaN
10   NaN  Chips  Bread  Butter   NaN  Apple
11  Wine    NaN    NaN  Butter  Milk  Apple
12  Wine  Chips  Bread  Butter  Milk    NaN
13  Wine    NaN  Bread     NaN  Milk  Apple
14  Wine    NaN  Bread  Butter  Milk  Apple
15  Wine  Chips  Bread  Butter  Milk  Apple
16   NaN  Chips  Bread  Butter  Milk  Apple
17   NaN  Chips    NaN  Butter  Milk  Apple
18  Wine  Chips  Bread  Butter  Milk  Apple
19  Wine    NaN  Bread  Butter  Milk  Apple
20  Wine  Chips  Bread     NaN  Milk  Apple
21   NaN  Chips    NaN     NaN  

In [2]:
records = []
for i in range(0, df.shape[0]):
    records.append([str(df.values[i, j]) for j in range(0, df.shape[1])])

In [3]:
#chuyển records thành transaction
te = TransactionEncoder()
te_ary = te.fit(records).transform(records)
df = pd.DataFrame(te_ary, columns=te.columns_)
print(df)

    Apple  Bread  Butter  Chips   Milk   Wine    nan
0    True   True    True   True   True   True  False
1   False   True    True  False   True   True   True
2   False   True    True  False   True  False   True
3    True  False   False   True  False  False   True
4    True   True    True   True   True   True  False
5   False  False   False   True   True   True   True
6    True   True    True   True  False   True   True
7   False  False   False   True   True   True   True
8    True   True   False  False  False   True   True
9   False   True    True  False   True   True   True
10   True   True    True   True  False  False   True
11   True  False    True  False   True   True   True
12  False   True    True   True   True   True   True
13   True   True   False  False   True   True   True
14   True   True    True  False   True   True   True
15   True   True    True   True   True   True  False
16   True   True    True   True   True  False   True
17   True  False    True   True   True  False 

In [11]:
min_support = 0.6
min_confidence = 0.8

In [12]:
F1 = []
for i in range(df.shape[1] - 1):
    F1.append([df.columns.tolist()[i], int((df.iloc[:, i]).sum()), float(int((df.iloc[:, i]).sum()) / df.shape[0])])

df1 = pd.DataFrame(F1, columns=['Item', 'Frequency', 'Support'])
df1

,Item,Frequency,Support
0,Apple,15,0.681818
1,Bread,16,0.727273
2,Butter,15,0.681818
3,Chips,14,0.636364
4,Milk,17,0.772727
5,Wine,16,0.727273


In [13]:
df1 = df1[df1['Support'] > min_support]
df1

,Item,Frequency,Support
0,Apple,15,0.681818
1,Bread,16,0.727273
2,Butter,15,0.681818
3,Chips,14,0.636364
4,Milk,17,0.772727
5,Wine,16,0.727273


In [14]:
F2 = []
for i in range(df.shape[1]):
    for j in range(i + 1, df.shape[1]):
        F2.append([df.columns.tolist()[i] + "," + df.columns.tolist()[j],
        int((df.iloc[:, i] & df.iloc[:, j]).sum()),
        float(int((df.iloc[:, i] & df.iloc[:, j]).sum())) / df.shape[0]])

df2 = pd.DataFrame(F2, columns=['Item', 'Frequency', 'Support'])
df2

,Item,Frequency,Support
0,"Apple,Bread",12,0.545455
1,"Apple,Butter",11,0.500000
2,"Apple,Chips",10,0.454545
3,"Apple,Milk",11,0.500000
4,"Apple,Wine",11,0.500000
5,"Apple,nan",11,0.500000
6,"Bread,Butter",13,0.590909
7,"Bread,Chips",9,0.409091
8,"Bread,Milk",13,0.590909
9,"Bread,Wine",13,0.590909


In [15]:
df2 = df2[df2['Support'] > min_support]
df2

,Item,Frequency,Support
18,"Milk,Wine",14,0.636364


In [17]:
support = pd.concat([df1, df2])
support

,Item,Frequency,Support
0,Apple,15,0.681818
1,Bread,16,0.727273
2,Butter,15,0.681818
3,Chips,14,0.636364
4,Milk,17,0.772727
5,Wine,16,0.727273
18,"Milk,Wine",14,0.636364


In [19]:
rules = []
if support.iloc[6, 2] / support.iloc[4, 2] > min_confidence:
    rules.append(['(Milk)', '(Wine)', df2.iloc[0, 2]])
if support.iloc[6, 2] / support.iloc[5, 2] > min_confidence:
    rules.append(['(Wine)', '(Milk)', df2.iloc[0, 2]])

rules = pd.DataFrame(rules, columns=['antecedents', 'consequents', 'support'])
rules

,antecedents,consequents,support
0,(Milk),(Wine),0.636364
1,(Wine),(Milk),0.636364
